---
title: Metamet. A Toolkit for Meteorological Data Standardisation  
short_title: Metamet Notebook  
description: metamet is an R package which attempts to solve many of the problems encountered in working with meteorological observation data.  
  
date: 2026-07-30  
updated: 2026-07-30  

license:  
  code: MIT  
  content: MIT + file LICENSE  

github: https://github.com/NERC-CEH/metamet  
thumbnail: "../images/edsb-favicon.png"  

authors:
  - name: Peter E. Levy  
    orcid: https://orcid.org/0000-0002-8505-1901  
    corresponding: true  
    email: plevy@ceh.ac.uk  
    affiliations:  
      - name: UK Centre for Ecology and Hydrology (UKCEH)  
        ror: https://ror.org/00pggkr55  
  - name: Claudia Caporusso  
    orcid: https://orcid.org/0009-0001-2852-4880  
    corresponding: false  
    email: clacap@ceh.ac.uk  
    affiliations:  
      - name: UK Centre for Ecology and Hydrology (UKCEH)  
        ror: https://ror.org/00pggkr55  
  - name: James M. Cash  
    orcid: https://orcid.org/0000-0002-8567-1377  
    corresponding: false  
    email: jamcas@ceh.ac.uk  
    affiliations:  
      - name: UK Centre for Ecology and Hydrology (UKCEH)  
        ror: https://ror.org/00pggkr55  
  - name: Karen Hei-Laan Yeung  
    orcid: https://orcid.org/0009-0006-0400-3980  
    corresponding: false  
    email: karung@ceh.ac.uk  
    affiliations:  
      - name: UK Centre for Ecology and Hydrology (UKCEH)  
        ror: https://ror.org/00pggkr55  

funding:
  - statement: This research was supported by NERC, through the UKCEH National Capability for UK Challenges Programme NE/Y006208/1.
    awards:
      - id: NE/Y006208/1
        name: UKCEH National Capability for UK Challenges Programme
        sources:
          - name: NERC
---

# Metamet. A Toolkit for Meteorological Data Standardisation  

`metamet` is an R package which attempts to solve many of the problems
encountered in working with meteorological observation data.
It provide a system for:

- standardising metadata
- converting between file formats
- converting between variable naming conventions
- converting units
- automating QA/QC
- facilitating manual QA/QC via a shiny app
- imputing missing values or "gap-filling"

It does this by defining:

1. a standardised generic data structure with enough complexity to hold both the 
observational data and the metadata, including site-specific, variable-specific 
and individual record-specific metadata; and
2. methods/functions for converting data between formats, combining data from 
different sources, quality control and gap-filling.

## Getting started with the Shiny app

`metamet` includes an interactive Shiny application for manual QA/QC and gap-filling of meteorological data. The app lets you build a `metamet` object from raw data files, inspect and correct observations interactively, and export the processed data.

### Install app dependencies

The app requires several additional packages that are not installed by default. Include them all at once with `pak`:

In [ ]:
``` {r, eval = FALSE}
install.packages("pak")
pak::pak("NERC-CEH/metamet", dependencies = TRUE)
```

: 

Or install them individually:

In [ ]:
``` {r, eval = FALSE}
install.packages(c(
  "shiny", "shinydashboard", "shinyjs", "shinyFiles",
  "shinyvalidate", "shinycssloaders", "ggiraph", "glue"
))
```

### Launch the app

In [ ]:
``` {r, eval = FALSE}
metamet::run_shiny()
```

### App workflow overview

The app guides you through a four-stage workflow:

1. **Create or open a `metamet` object** – Use the *Create new Metamet object*
   wizard to import a raw data file (CSV, Campbell TOA5, old Campbell `.dat`/`.dld`,
   or CEDA BADC-CSV), map variables to ICOS names, set QC ranges, and optionally
   attach ERA5 reference data. The wizard saves the result as a `.rds` file.
   Alternatively, use *Open existing Metamet object* to load a previously saved `.rds`.

2. **Select date range and QA/QC** – Choose a start and end date/time, click
   *Retrieve from database*, then inspect each variable in its own interactive
   plot tab. Select suspect data points with the lasso tool, pick a gap-filling
   method (time interpolation, regression, ERA5 substitution, or others), add an
   optional comment explaining the change, and click *Impute selection*. Click
   *Finished checking variable* when a variable is signed off.

3. **Save changes** – Click *Save changes* to write a new `.rds` file (named with
   your system username and today's date) plus a CEDA-formatted output in the same
   directory as the source file.

4. **Download processed data** – Export Level 1, Level 2, or CEDA-formatted data
   as `.csv` (or `.zip` for Level 1/2) from the *Download processed data* tab.

For a detailed walkthrough see the
[App User Guide](https://nerc-ceh.github.io/metamet/articles/app_user_guide.html).

## Basic metamet workflow - create a metamet object without the app

Basic usage is to first create `metamet` objects from files or pre-existing data 
frames or data tables. Because all the metadata describing the observations is
available in the structure, objects can be processed relatively easily so as to:

- rename variables in standard naming conventions
- combine data from different sites
- apply quality control
- impute missing data.

In [ ]:
```{r example}
library(metamet)
fname_dt <- testthat::test_path("data-raw/UK-AMO/UK-AMO_BM_dt_2026.csv")
fname_meta <- testthat::test_path("data-raw/dt_meta.xlsx")
fname_site <- testthat::test_path("data-raw/dt_site.csv")

mm <- metamet(
  dt = fname_dt,
  dt_meta = fname_meta,
  dt_site = fname_site,
  site_id = "UK-AMO"
)

# print the outline strucutre:
mm
```


A typical workflow would go on to perform tasks such as adding reference data 
from ECMWF ERA5 reanalysis, join with other `metamet` objects, apply quality 
control algorithms, impute missing values by various algorithms, and check the 
data manually for additional QC. This is illustrated  below.

In [ ]:
```{r workflow, eval = FALSE}
mm <- add_era5(
  mm,
  fname_era5 = testthat::test_path("data-raw/dt_era5.csv")
)
mm <- join(mm, mm_old)
mm <- apply_qc(mm)
mm <- impute(mm = mm)
run_shiny()
```


Clearly a two-dimensional data table is not sufficient to hold all the 
information. Instead we define a `metamet` data object as a set of related data 
tables. We implement this as a list in R, containing five data tables (prefix `dt_`) 
explained in the table below.

| Name | Type | Contains | Rows correspond to | Columns correspond to |
|---|---|---|---|---|
| dt | data.table | sensor data | time intervals | variables |
| dt_meta | data.table | variable- and time-specific meta data (like netCDF data attributes) e.g. coords for sensor locations | variables x time period | metadata variables |
| dt_site | data.table | site-specific meta data (like netCDF global attributes) | sites | metadata variables |
| dt_qc | data.table | QC codes | time intervals | variables |
| dt_ref | data.table | ref data e.g. era5 | time intervals | variables |


The following provides an example of how to structure the rest of your notebook. Use headings, sub-headings, and a mix of code cells and markdown cells to guide others through your method step-by-step.

## Met Data Standards

The most basic problem is simply of agreeing on standard names for variables.
Several conventions exist, but none are widely used outside the modelling
community. Some naming conventions are listed below.
  

| Name | Defines | Link | Issues |
|---|---|---|---|
| Climate Forecast (CF) | Names & netCDF file format | [link](https://cfconventions.org/Data/cf-standard-names/docs/guidelines.html), and [from CEDA](https://help.ceda.ac.uk/article/4507-the-cf-metadata-convention) |
| ICOS | Variable names | [link](https://www.icos-etc.eu/icos/documents/instructions/bmform) | |
| ERA5 | Variable names | [link](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=documentation) | |
| AmeriFlux BADM | Variable names | [link](/ameriflux.lbl.gov/data/badm/badm-standards/) | Does not including most met variables |
| Copernicus Station Exchange Format (SEF) | Variable names | [link](https://datarescue.climate.copernicus.eu/variablenames) | |
| Copernicus Station Exchange Format (SEF) | Plain text file format  | [link](https://datarescue.climate.copernicus.eu/station-exchange-format-sef) | Only one variable per file limits use |
| CEDA BADC-CSV | Plain text file format  | [link](https://help.ceda.ac.uk/article/105-badc-csv) | Format not easily machine-readable |

  
These conventions are not systematically constructed, so are not
easily extended or modified to be more generally useable; several are not even 
easily machine-readable.
For example, the Climate Forecast (CF) convention constructs a set of [standard
names](https://cfconventions.org/Data/cf-standard-names/docs/guidelines.html) 
used widely in climate modelling. Users are able to suggest new variables names 
in an ad hoc manner. The result is there are ~5250 arbitrarily defined variable 
names, with no system to their naming. For example, there is no system to the 
naming to group together different types of temperature measurements (such as soil 
temperature, surface temperature, air temperature at 2 m or 10 m).
Eleven chemical species are defined, but arbitraryily, so no other chemical 
species can be referred to without inventing new names.

Because none of these is widely used in the measurement community, we need to 
build the flexibility to: 

- allow easy conversion of ad hoc local naming schemes into one or more standards, and 
- translate easily between different existing standards (e.g. ERA5 to CF-1).

## Workflow Example 1

This vignette illustrates the typical workflow for processing met data with 
`metamet`. The basic steps are to:

- read the observation data
- create a corresponding metadata table `dt_meta` for the observed variables 
- create a metadata table `dt_site` for each site with observations
- combine these into a single `metamet` object

## Example data

To illustrate, we will use some publicy available data from Whim Moss, available
from the [EIDC data centre](https://catalogue.ceh.ac.uk/documents/ba9ecad2-1740-41e6-8f73-b308340d49fe).
Metadata are available in the supporting documentation, but not in a 
machine-readable format, so we have to extract it manually the first time. 
Thereafter it is available within the `metamet` object for all future use.

Firstly, we load the `metamet` library, as well as the `here` library which 
simplifies specifying file paths.

In [ ]:
```{r}
here::i_am("vignettes/workflow_example.Rmd")
library(metamet)
library(here)
```

### Observation data `dt`

Having downloaded the observation data, we can read it from a file and display 
a few rows.

In [ ]:
```{r}
fname <- here("inst/extdata/UK-WHM/historical/eidc/whim_met_2002_2023.csv")

dt <- data.table::fread(fname)
dim(dt)
dt
```

The data consist of a timestamp column, and 13 variables observed every 15 
minutes from 2003 until 2023, giving `r nrow(dt)` rows and `r ncol(dt)` columns.

### Observation metadata `dt_meta`

In [ ]:
```{r, include = FALSE}
fname <- here("inst/extdata/dt_meta.csv")
dt_meta <- data.table::fread(fname)

v_col <- c(
  "site",
  "name_dt",
  "name_local",
  "units_local",
  "type",
  "time_char_format",
  # "start_date",
  # "end_date",
  # "horizontal_id",
  # "vertical_id",
  # "replicate_id",
  # "gf_time",
  # "gf_era5",
  # "gf_night_zero",
  # "diff_ref_max",
  # "sensor_make",
  # "sensor_model",
  "range_min",
  "range_max",
  "name_era5",
  "units_era5",
  "imputation_method"
)
```

In [ ]:
```{r}
knitr::kable(dt_meta[site == "UK-WHM", ..v_col], format = "html")
```

### Site data `dt_site`

The site metadata required is minimal. Any additional variables can be added, 
but at a minimum we require the site name, a uniquely identifying code `site`,
longitude, latitude (in degrees with a decimal fraction) and elevation (in m).
These are easily gleaned from the supporting documentation and could be entered
in excel, read in as a .csv text file, or entered directly in R as below.

In [ ]:
```{r, fig.show='hold'}
dt_site <- data.table::data.table(
  site = "UK-WHM",
  long_name = "Whim Moss",
  lon = -3.27155,
  lat = -55.76566,
  elev = 316
)
dt_site
```

In practice, we are likely to have multiple sites as in the table below, and it
is easiest to append rows to a .csv or excel file as new sites are added.

In [ ]:
```{r, fig.show='hold'}
fname <- here("inst/extdata/dt_site.csv")
dt_site <- data.table::fread(fname)
knitr::kable(dt_site, format = "html")
```

## Workflow Example 2

This vignette illustrates the typical workflow for quality control of met data 
from multiple sites simultaneously with `metamet`. We assume that the basic 
steps have been carried out already: reading the raw observation data, creating 
systematic metadata for the variables and each site (described 
in [Workflow Example 1](workflow_example.html)).
This produces a single `metamet` object for each site.

First, we load the `metamet` library, and read the data for each of three sites
from file. 

In [ ]:
```{r}
here::i_am("vignettes/workflow_example.Rmd")
library(here)
library(ggplot2)
library(metamet)

mm_amo <- readRDS(
  file = here::here("inst/extdata/UK-AMO/UK-AMO_BM_mm_2023.rds")
)
mm_ebu <- readRDS(
  file = here::here("inst/extdata/UK-EBU/UK-EBU_BM_mm_2023.rds")
)
mm_whm <- readRDS(
  file = here::here("inst/extdata/UK-WHM/UK-WHM_BM_mm_2023.rds")
)
```

This gives us all the half-hourly data for 2023 from each site.  If we want to
process a shorter time period, we can subset as required with `subset_by_date`.
Here we look at two days just for speed.

In [ ]:
```{r}
mm_amo <- subset_by_date(mm_amo, "2023-09-01", "2023-09-03")
mm_ebu <- subset_by_date(mm_ebu, "2023-09-01", "2023-09-03")
mm_whm <- subset_by_date(mm_whm, "2023-09-01", "2023-09-03")
```

In order to combine data from different sites in a meningful way, they all need
to use the same naming convention. Here we convert two of the sites to the ICOS
convention using the `change_naming_convention` function below (UK-AMO data are
already produced in this form).

In [ ]:
```{r}
mm_ebu <- change_naming_convention(mm_ebu, name_convention = "name_icos")
mm_whm <- change_naming_convention(mm_whm, name_convention = "name_icos")
```

Although the base names of variable are now standardised across sites, the 
specific variables measured will vary across sites. For example, one site may
have two replicate measurements of air temperature TA_4_1_1 and TA_5_1_1, while
another has a single replicate TA_1_1_1. These cannot be combined as a single 
variable (or related variables) in wide format. Also, wide format would require
columns for the full set of variables found across all sites, many of which will
be empty for all sites except one. The efficient solution is to convert to long
format, so all values are in a single column, with additional columns to 
identify the site, timestamp, varaiable type and specific varaiable name. We do
this with the `reshape_wide_to_long` function, shown below.

In [ ]:
```{r}
mm_amo <- metamet_reshape(mm_amo, "long")
mm_ebu <- metamet_reshape(mm_ebu, "long")
mm_whm <- metamet_reshape(mm_whm, "long")
```

We can now see the structure of UK-AMO data 

In [ ]:
```{r}
mm_amo$dt
```

and see it is identical to the structure of UK-EBU (and UK-WHM) data.

In [ ]:
```{r}
mm_ebu$dt
```

Given this structure, we can simply append all the rows for all data tables to
create a single `metamet` object.

In [ ]:
```{r}
mm <- rbind_metamet(
  mm_amo,
  l_dt = list(mm_amo$dt, mm_ebu$dt, mm_whm$dt),
  l_dt_meta = list(mm_amo$dt_meta, mm_ebu$dt_meta, mm_whm$dt_meta),
  l_dt_site = list(mm_amo$dt_site, mm_ebu$dt_site, mm_whm$dt_site)
)
```

We can now plot all the variables of a given type for all the types, identifying
site by colour or separate panels (facets). Below we plot soil temperature 
against time; the solid black line shows the reference data, which in this case 
is ERA5 reanalysis data for the grid cell containing the site.

In [ ]:
```{r}
p <- ggplot(
  mm$dt[name_icos == "TS", ],
  aes(TIMESTAMP, value, colour = var_name)
)
p <- p + geom_line(aes(y = ref), colour = "black")
p <- p + geom_point()
p <- p + facet_wrap(~site)
print(p)
```

We can also plot all variable against the reference (ERA5) data to check for
anomalous deviations from the expected relationship.

In [ ]:
```{r}
p <- ggplot(
  mm$dt[name_icos == "TA", ],
  aes(ref, value, colour = var_name)
)
p <- p + geom_abline()
p <- p + geom_point()
p <- p + facet_wrap(~site)
print(p)
```

### Using Openair Mapping Functions

This vignette illustrates the use of `openairmaps` to produce a simple 
interactive leaflet maps showing weather station sites.


## Network map

The simplest function is simply to create a network map with the `network_map` 
function shown below

In [ ]:
```{r}
here::i_am("vignettes/openairmaps.Rmd")
library(metamet)

fname_site <- here::here("data-raw/dt_site.csv")
dt_site <- data.table::fread(fname_site)
network_map(dt_site)
```

## Polar plot maps

We can also create interactive polar plot maps using `polar_map()`.
The function extracts the appropriate time, wind speed, and wind direction 
fields from a metamet object, and produces a leaflet-based polar map.
for a varible `var_name` chosen to be mapped.

In [ ]:
```{r}
fname_dt <- here::here("data-raw/UK-AMO/UK-AMO_BM_dt_2026.csv")
  fname_meta <- here::here("data-raw/dt_meta.xlsx")
  fname_site <- here::here("data-raw/dt_site.csv")
  # half-hourly data
  mm <- metamet(
    dt = fname_dt,
    dt_meta = fname_meta,
    dt_site = fname_site,
    site_id = "UK-AMO"
  )
  polar_map(mm, var_name = "PA_4_1_1")
```

### MetQC App User Guide

This document explains how to use the MetQC Shiny app bundled with the `metamet`
package. It covers every tab in the app, all major controls, and includes a
step-by-step FAQ for common tasks.

## Prerequisites

The app requires a set of optional packages that are not installed automatically.
Install them before launching:

In [ ]:
```r
# Recommended: install metamet plus all optional dependencies in one step
install.packages("pak")
pak::pak("NERC-CEH/metamet", dependencies = TRUE)
# Or install the app packages individually
install.packages(c(
  "shiny", "shinydashboard", "shinyjs", "shinyFiles",
  "shinyvalidate", "shinycssloaders", "ggiraph", "glue"
))
```

Launch the app from R:

In [ ]:
```r
metamet::run_shiny()
```

The app opens in your default web browser. It will stop automatically when you
close the browser tab or click **Stop App** in the left sidebar.

---

## App layout

The left sidebar contains five items:

| Sidebar item | Purpose |
|---|---|
| Create new Metamet object | Six-step wizard to build a `.rds` from raw data files |
| Open existing Metamet object | Browse and load an existing `.rds` |
| Select date range and QA/QC | Main dashboard for interactive data validation |
| Download processed data | Export processed data as CSV or ZIP |
| Help and Documentation | Links to gap-fill methods and this guide |

Your system username (from `Sys.info()[["user"]]`) is used automatically to
label any changes you make. It is not currently configurable within the app.

---

## Tab 1 — Create new Metamet object

This six-step wizard takes you from a raw data file to a saved `metamet` `.rds`
file ready for QA/QC. Each step must be completed in order; use the **Back**
and **Continue** buttons to navigate.

### Step 1: Load data file

Select the format of your raw data file:

- **Plain CSV** — a standard CSV file with a header row.
- **Campbell TOA5** — data logger output in TOA5 (ASCII) format.
- **Old Campbell (.dat + .dld)** — older Campbell files; you will be prompted to
  select both the `.dat` data file and the `.dld` metadata file.
- **CEDA BADC-CSV** — the BADC-CSV format used by CEDA data archives.

Click **Select data file**, browse to your file, then click **Load & preview**.
A preview table will appear. If the file contains multiple tables (Campbell TOA5),
a drop-down lets you choose which table to use. Select the column that contains
the timestamp, then click **Continue**.

### Step 2: Site information

Provide site-level metadata. You can either:

- **Enter manually** — fill in the Site ID, site name, latitude, longitude,
  elevation, and the date range over which this metadata applies.
- **Load from CSV file** — select an existing `dt_site` CSV and choose the row
  corresponding to your site.

Click **Continue to variable mapping** when ready.

### Step 3: Map data columns to ICOS variable names

Each column in your data file must be mapped to a standard variable name. You can:

- **Map manually** — for each data column, select the matching ICOS variable from
  a drop-down list. Columns not mapped to any variable can be left as *— skip —*.
- **Load dt_meta from CSV file** — if you already have a `dt_meta` metadata file,
  load it here and the mappings will be applied automatically.

Click **Continue** when all required columns are mapped.

### Step 4: Set units and QC ranges *(optional)*

For each mapped variable you can set:

- **Units** — the physical units of the raw data.
- **Valid range** — minimum and maximum acceptable values for automatic range checking.
- **Imputation method** — the default gap-filling algorithm to apply to this variable.

This step can be skipped; settings can be refined later. Click **Continue**.

### Step 5: ERA5 reference data *(optional)*

Attach ERA5 reanalysis data as a reference dataset (`dt_ref`). Select an ERA5
CSV file (as exported by `add_era5()`). The reference data will be plotted
alongside observations in the QA/QC dashboard. Click **Continue** (or skip).

### Step 6: Review and save

A summary of the object is displayed. Click **Download as .rds** to save the
`metamet` object to a file. This file is the input for the QA/QC dashboard.

---

## Tab 2 — Open existing Metamet object

Click **Browse for .rds file**, navigate to a previously saved `metamet` `.rds`
file, and select it. The file is loaded and the app switches automatically to
the **Select date range and QA/QC** tab. A notification confirms the file name
that was loaded.

The `.rds` file must be a `metamet` object in long format (or wide format — the
app reshapes it automatically).

---

## Tab 3 — Select date range and QA/QC

This is the main validation interface.

### Selecting a date range

The **Data Selection** box contains:

- **Start date / End date** — date pickers pre-filled with the earliest and latest
  timestamps in the loaded file.
- **Hour / Minute** — fine-tune the start and end times within the selected days.

Click **Retrieve from database** to extract the chosen time window. The
*Extracted Data* panel appears below.

After retrieval, the **Compare variables** button becomes active. Click it to
open a scatter-plot modal comparing any two variables in the extracted window.

### Extracted data panel

The panel contains one tab per variable. Each tab shows:

- **Replicate checkboxes** *(if the variable has multiple replicates)* — tick/untick
  individual sensor replicates to show or hide them on the plot.
- **Interactive plot** — a time-series plot produced by `ggiraph`. Point colours
  indicate the current QC code (grey = valid, coloured = imputed or flagged).
  Hover over a point to see its timestamp and value.
- **Rescale reference to observations** checkbox — when ERA5 reference data is
  present, tick this to rescale the ERA5 values to the observation range before
  plotting.
- **Point size** slider — adjust the visual size of plotted points (0.5 – 8).

### Imputing (gap-filling) data

1. Move the mouse over the plot area. A toolbar appears in the top-right corner
   of the plot. Click the **lasso selection** button (the left-most icon).
2. Draw a lasso around the data points you want to replace. Selected points turn
   red.
3. Choose a **Gap-Filling Method** from the drop-down below the plot. Available
   methods are described in the *Help and Documentation* tab (and in the
   `gap_fill_methods` vignette):
   - **Time interpolation** — GAM spline through time; a smoothness slider appears.
   - **Regression** — linear regression against another variable; a covariate
     drop-down appears.
   - **ERA5** — substitute values from the reference dataset.
   - **Zero** — set selected values to zero.
   - **Non-negative** — set negative values to zero.
   - **Night zero** — set night-time values to zero.
4. *(Optional)* Type a brief **reason for imputation** in the comment box.
5. Use the **Do not alter data estimated by** checkboxes to protect any previously
   imputed points from being overwritten.
6. Click **Impute selection**. The plot updates immediately.
7. Repeat steps 1–6 as needed for the same or other variables.
8. When a variable looks correct, click **Finished checking variable for date range**.
   The variable's tab turns grey to indicate it has been signed off.

### Saving changes

Click **Save changes** when you are satisfied with all variables in the selected
date range.

> **Important:** unsaved changes are lost if you click *Retrieve from database*
> again for a new date range. Always save before moving to a new window.

The app writes two new files next to the source `.rds`:

- `<original_name>_qc_by_<username>_on_<date>.rds` — full `metamet` object with
  updated QC codes and imputed values.
- `<original_name>_qc_by_<username>_on_<date>_ceda.rds` — CEDA-formatted output.

A notification confirms the files were created.

Click **Restart app** at any time to reload the app and start fresh.

---

## Tab 4 — Download processed data

Select the format from the drop-down:

| Option | Output | Contents |
|---|---|---|
| Level 1 | `.zip` containing two `.csv` files | Raw data (`dt`) and QC codes (`dt_qc`) |
| Level 2 | `.zip` containing two `.csv` files | QC-processed data and updated QC codes |
| CEDA | Single `.csv` file | CEDA BADC-CSV formatted output |

Click **Download** to save the file.

---

## Tab 5 — Help and Documentation

Sub-items link to:

- **Gap-fill methods** — descriptions of each imputation algorithm.
- **App guide** — this document.
- **Data process guide** — overview of the data processing pipeline.

---

## Frequently asked questions

#### Q: How do I validate data step by step?

1. Load a `.rds` file via *Open existing Metamet object* (or create one first).
2. On the *Select date range and QA/QC* tab, choose your start and end date/time.
3. Click **Retrieve from database**.
4. Work through each variable tab: inspect the plot, impute suspect points as
   needed, and click **Finished checking variable** to sign off.
5. Click **Save changes** to write the output files.

#### Q: How do I change the username the app uses?

The username is read automatically from your operating system account
(`Sys.info()[["user"]]`) and is not currently configurable within the app.

#### Q: I want to download the data

Click the *Download processed data* tab in the left sidebar. Choose Level 1,
Level 2, or CEDA from the drop-down and click **Download**.

Level 1 and Level 2 outputs are `.zip` archives containing a data CSV and a QC
CSV. CEDA output is a single BADC-CSV file.

#### Q: How do I create a metamet object from scratch (new site / new year)?

Use the *Create new Metamet object* wizard (first item in the sidebar). Work
through the six steps to load your raw data, enter site information, map
variables, set QC ranges, attach ERA5 data, and download the `.rds`.

#### Q: The app is slow or has frozen — what should I do?

Click **Restart app** (visible in the *Extracted Data* panel after data
retrieval) or close the browser tab and re-run `metamet::run_shiny()`.

#### Q: My question is not listed here, I have noticed a mistake, or I would like a new feature

log a [GitHub issue](https://github.com/NERC-CEH/metamet/issues), or email
Pete (plevy@ceh.ac.uk).